# XML Reading and Memory Safety

This notebook exercises `to_pyarrow` with XML whole-document rows and explicit row-tag streaming. It also shows the same XML row option through a file-to-file converter and demonstrates the configured memory guard.

Create a small XML file with repeated `<record>` elements. The file is local, and generated outputs stay under `examples/files/06_xml_reading_and_memory`.

In [ ]:
from pathlib import Path

import schema_sanitizer as ss

out_dir = Path("files/06_xml_reading_and_memory/exercise_01_xml_rows")
out_dir.mkdir(parents=True, exist_ok=True)
xml_path = out_dir / "records.xml"
xml_path.write_text(
    """<records>
  <record id="1"><name>Ana</name><tag>new</tag><tag>priority</tag></record>
  <record id="2"><name>Alex</name><value>19.5</value></record>
</records>""",
    encoding="utf-8",
)
xml_path

Without `xml_row_tag`, XML input treats the whole document as one row, matching the behavior of a single JSON object.

In [ ]:
document_result = ss.to_pyarrow(xml_path, input_format="xml")
document_result.clean_data.to_pylist()

With `xml_row_tag='record'`, direct child `<record>` elements stream as separate rows. the chunk size derived from `memory_limit_bytes` can be smaller than a row; the scanner keeps only the active XML window until a complete row is available.

In [ ]:
row_result = ss.to_pyarrow(
    xml_path,
    input_format="xml",
    xml_row_tag="record",
    memory_limit_bytes=16 * 1024 * 1024,
)
row_result.clean_data.to_pylist()

The same row-tag option works when XML is the input format for a converter.

In [ ]:
jsonl_path = out_dir / "records.sanitized.jsonl"
ss.to_jsonl(
    xml_path,
    jsonl_path,
    input_format="xml",
    xml_row_tag="record",
    memory_limit_bytes=16 * 1024 * 1024,
)
jsonl_path.read_text(encoding="utf-8").splitlines()

XML directory mode reads direct `.xml` children in filename order. Each file is one XML document row.

In [ ]:
folder_path = out_dir / "record_docs"
folder_path.mkdir(exist_ok=True)
(folder_path / "b.xml").write_text(
    '<record id="2"><name>Bea</name></record>', encoding="utf-8"
)
(folder_path / "a.xml").write_text(
    '<record id="1"><name>Ana</name></record>', encoding="utf-8"
)

folder_result = ss.to_pyarrow(folder_path, input_format="xml", input_mode="directory")
folder_result.clean_data.to_pylist()

A row-tag XML file can be larger than `memory_limit_bytes` when each active row window fits inside the budget.

In [ ]:
stream_path = out_dir / "many_orders.xml"
stream_path.write_text(
    "<records>"
    + "".join(f"<record><id>{i}</id><value>{i * 10}</value></record>" for i in range(40))
    + "</records>",
    encoding="utf-8",
)
stream_result = ss.to_pyarrow(
    stream_path,
    input_format="xml",
    xml_row_tag="record",
    memory_limit_bytes=16 * 1024 * 1024,
)
{"file_size": stream_path.stat().st_size, "rows": stream_result.clean_data.num_rows}

A single row that exceeds the active XML memory budget is rejected before it can be materialized.

In [ ]:
huge_row_path = out_dir / "huge_order.xml"
huge_row_path.write_text(
    "<records><record><payload>" + ("x" * (2 * 1024 * 1024)) + "</payload></record></records>",
    encoding="utf-8",
)

try:
    ss.to_pyarrow(
        huge_row_path, input_format="xml", xml_row_tag="record", memory_limit_bytes=1024 * 1024
    )
except (ss.SchemaSanitizerResourceError, ss.SchemaSanitizerOutOfMemoryError) as exc:
    print(exc.code.value, exc.detail)